In [7]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv('global_bleaching_environmental.csv')

/var/folders/6z/zp8_cpcs5sl5jpjff7w6dm300000gn/T/ipykernel_78256/113590985.py:1: DtypeWarning: Columns (13,15,24) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('global_bleaching_environmental.csv')


In [5]:
df.head()

,Site_ID,Sample_ID,Data_Source,Latitude_Degrees,Longitude_Degrees,Ocean_Name,Reef_ID,Realm_Name,Ecoregion_Name,Country_Name,...,TSA_FrequencyMax,TSA_FrequencyMean,TSA_DHW,TSA_DHW_Standard_Deviation,TSA_DHWMax,TSA_DHWMean,Date,Site_Comments,Sample_Comments,Bleaching_Comments
0,2501,10324336,Donner,23.163,-82.5260,Atlantic,nd,Tropical Atlantic,Cuba and Cayman Islands,Cuba,...,5,0,0,0.74,7.25,0.18,2005-09-15,nd,nd,nd
1,3467,10324754,Donner,-17.575,-149.7833,Pacific,nd,Eastern Indo-Pacific,Society Islands French Polynesia,French Polynesia,...,4,0,0.26,0.67,4.65,0.19,1991-03-15,The bleaching does not appear to have gained ...,The bleaching does not appear to have gained ...,nd
2,1794,10323866,Donner,18.369,-64.5640,Atlantic,nd,Tropical Atlantic,Hispaniola Puerto Rico and Lesser Antilles,United Kingdom,...,7,0,0,1.04,11.66,0.26,2006-01-15,nd,nd,nd
3,8647,10328028,Donner,17.760,-64.5680,Atlantic,nd,Tropical Atlantic,Hispaniola Puerto Rico and Lesser Antilles,United States,...,4,0,0,0.75,5.64,0.2,2006-04-15,nd,nd,nd
4,8648,10328029,Donner,17.769,-64.5830,Atlantic,nd,Tropical Atlantic,Hispaniola Puerto Rico and Lesser Antilles,United States,...,5,0,0,0.92,6.89,0.25,2006-04-15,nd,nd,nd


In [10]:
df = pd.read_csv("global_bleaching_environmental.csv")
df.replace("nd", np.nan, inplace=True)
df["Percent_Bleaching"] = pd.to_numeric(df["Percent_Bleaching"], errors="coerce")
df["Bleaching_YN"] = (df["Percent_Bleaching"] > 0).astype(int)

/var/folders/6z/zp8_cpcs5sl5jpjff7w6dm300000gn/T/ipykernel_78256/788732504.py:1: DtypeWarning: Columns (13,15,24) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("global_bleaching_environmental.csv")


In [14]:
features = ["TSA", "TSA_DHW", "SSTA", "ClimSST",
            "Windspeed", "Turbidity", "Cyclone_Frequency",
            "Distance_to_Shore", "Depth_m"]

for f in features:
    df[f] = pd.to_numeric(df[f], errors="coerce")

df = df[features + ["Bleaching_YN"]].dropna()

X = df[features].values.astype(float)
y = df["Bleaching_YN"].values.astype(float).reshape(-1, 1)

print("Class balance:")
print(pd.Series(y.flatten()).value_counts())

Class balance:
0.0    23245
1.0    16177
dtype: int64


In [ ]:
# normalize
X = (X - X.mean(axis=0)) / X.std(axis=0)

# train test split
n = len(X)
idx = np.random.permutation(n)
t, v = int(0.7*n), int(0.85*n)
X_train, y_train = X[idx[:t]],  y[idx[:t]]
X_val,   y_val   = X[idx[t:v]], y[idx[t:v]]
X_test,  y_test  = X[idx[v:]],  y[idx[v:]]

In [ ]:
 # init weights
np.random.seed(42)
n_in, n_h1, n_h2, n_out = X.shape[1], 64, 32, 1

W1 = np.random.randn(n_in, n_h1) * 0.01;  b1 = np.zeros((1, n_h1))
W2 = np.random.randn(n_h1, n_h2) * 0.01;  b2 = np.zeros((1, n_h2))
W3 = np.random.randn(n_h2, n_out) * 0.01; b3 = np.zeros((1, n_out))

def relu(z):    return np.maximum(0, z)
def sigmoid(z): return 1 / (1 + np.exp(-z))
def d_relu(z):  return (z > 0).astype(float)

In [ ]:
def forward(X):
    z1 = X  @ W1 + b1;  a1 = relu(z1)
    z2 = a1 @ W2 + b2;  a2 = relu(z2)
    z3 = a2 @ W3 + b3;  a3 = sigmoid(z3)
    return z1, a1, z2, a2, z3, a3

def bce(y, yhat):
    yhat = np.clip(yhat, 1e-9, 1 - 1e-9)
    return -np.mean(y * np.log(yhat) + (1 - y) * np.log(1 - yhat))

lr = 0.01
train_losses, val_losses = [], []

for epoch in range(500):
    z1, a1, z2, a2, z3, a3 = forward(X_train)

    dz3 = a3 - y_train
    dW3 = a2.T @ dz3 / len(X_train);  db3 = dz3.mean(axis=0, keepdims=True)
    dz2 = (dz3 @ W3.T) * d_relu(z2)
    dW2 = a1.T @ dz2 / len(X_train);  db2 = dz2.mean(axis=0, keepdims=True)
    dz1 = (dz2 @ W2.T) * d_relu(z1)
    dW1 = X_train.T @ dz1 / len(X_train); db1 = dz1.mean(axis=0, keepdims=True)

    W3 -= lr*dW3; b3 -= lr*db3
    W2 -= lr*dW2; b2 -= lr*db2
    W1 -= lr*dW1; b1 -= lr*db1

    train_losses.append(bce(y_train, forward(X_train)[-1]))
    val_losses.append(bce(y_val, forward(X_val)[-1]))

    if epoch % 50 == 0:
        print(f"Epoch {epoch} | train loss: {train_losses[-1]:.4f} | val loss: {val_losses[-1]:.4f}")


In [ ]:
y_pred = (forward(X_test)[-1] >= 0.5).astype(int)
acc = np.mean(y_pred == y_test)
print(f"\nTest accuracy: {acc:.4f}")

Epoch 0 | train loss: 0.6931 | val loss: 0.6931
Epoch 50 | train loss: 0.6894 | val loss: 0.6894
Epoch 100 | train loss: 0.6866 | val loss: 0.6865
Epoch 150 | train loss: 0.6844 | val loss: 0.6842
Epoch 200 | train loss: 0.6827 | val loss: 0.6825
Epoch 250 | train loss: 0.6814 | val loss: 0.6811
Epoch 300 | train loss: 0.6803 | val loss: 0.6800
Epoch 350 | train loss: 0.6795 | val loss: 0.6792
Epoch 400 | train loss: 0.6789 | val loss: 0.6785
Epoch 450 | train loss: 0.6784 | val loss: 0.6780

Test accuracy: 0.5822
